# VOT Notebook Utilities Demo

This notebook demonstrates the main functionality in `vot.utilities.notebook`:

- notebook environment checks,
- workspace loading,
- running experiments,
- running analyses,
- visualizing sequences, tracker runs, and stored results.

Set the workspace path in Cell 3, then run cells in order.

In [ ]:
from pathlib import Path

from vot.workspace import Workspace
import vot.utilities.notebook as vnb

## Configuration

Set `WORKSPACE_PATH` to your workspace directory.

Optional knobs:

- `TRACKER_IDS`: explicit tracker references (for example `["mytracker"]`)
- `EXPERIMENT_ID`: explicit experiment identifier
- `SEQUENCE_NAME`: explicit sequence name

In [ ]:
WORKSPACE_PATH = "TODO: set workspace path here"
TRACKER_IDS = []
EXPERIMENT_ID = ""
SEQUENCE_NAME = ""

workspace_root = Path(WORKSPACE_PATH)
if not workspace_root.exists():
    raise FileNotFoundError(f"Workspace path does not exist: {WORKSPACE_PATH}")

workspace = Workspace.load(str(workspace_root))
print("Loaded workspace:", workspace_root)
print("Experiments:", len(workspace.stack))
print("Sequences:", len(workspace.dataset))

results_storage = workspace.storage.substorage("results")

if TRACKER_IDS:
    trackers = workspace.registry.resolve(*TRACKER_IDS, storage=results_storage, skip_unknown=False)
else:
    trackers = workspace.list_results(workspace.registry)

if not trackers:
    raise RuntimeError("No trackers resolved. Set TRACKER_IDS or run evaluation first.")

if EXPERIMENT_ID is None:
    experiment = workspace.stack[0]
else:
    matches = [e for e in workspace.stack if e.identifier == EXPERIMENT_ID]
    if not matches:
        raise RuntimeError(f"Experiment not found: {EXPERIMENT_ID}")
    experiment = matches[0]

if SEQUENCE_NAME is None:
    sequence = workspace.dataset[0]
else:
    matches = [s for s in workspace.dataset if s.name == SEQUENCE_NAME]
    if not matches:
        raise RuntimeError(f"Sequence not found: {SEQUENCE_NAME}")
    sequence = matches[0]

print("Selected tracker count:", len(trackers))
print("Selected experiment:", experiment.identifier)
print("Selected sequence:", sequence.name)

## SequenceView

Display a sequence frame with ground truth and an optional overlay region.

In [ ]:
view = vnb.SequenceView(sequence)
view.set_frame(0)
view.show()

## Run Experiment

Run one experiment for selected trackers and sequences.

By default this is disabled to avoid accidental long runs.

In [ ]:
vnb.run_experiment(
    experiment=experiment,
    sequences=[sequence],
    trackers=[trackers[0]],
    force=False,
    persist=True,
)

## 3) Run Analysis

Compute stack analyses and return in-memory results.

In [ ]:
analysis_results = vnb.run_analysis(
    workspace=workspace,
    trackers=[trackers[0]],
    sequences=[sequence.name],
    experiments=[experiment.identifier],
)

print("run_analysis returned type:", type(analysis_results))

## 4) Optional Analysis Serialization

Save analysis output to workspace storage in JSON or YAML.

In [ ]:
RUN_SERIALIZE = False
SERIALIZE_FORMAT = "json"
SERIALIZE_NAME = None

if RUN_SERIALIZE:
    serialized = vnb.run_analysis(
        workspace=workspace,
        trackers=[trackers[0]],
        sequences=[sequence.name],
        experiments=[experiment.identifier],
        output_format=SERIALIZE_FORMAT,
        name=SERIALIZE_NAME,
    )
    print("Serialized:", serialized)
else:
    print("Set RUN_SERIALIZE = True to write analysis files to workspace/analysis.")

## Visualize Stored Results

Overlay tracker trajectories from stored experiment results.

In [ ]:
vnb.visualize_results(experiment=experiment, sequence=sequence, trackers=[trackers[0]])

## 6) Live Tracker Visualization

Runs the tracker interactively frame-by-frame. This may start external tracker processes.

In [ ]:
RUN_LIVE_TRACKER = False

if RUN_LIVE_TRACKER:
    vnb.visualize_tracker(trackers[0], sequence)
else:
    print("Set RUN_LIVE_TRACKER = True to open interactive tracker playback widget.")